# 病歷守門人（Colab 版）：用免費 T4 微調 Gemma 4 E2B 找出病歷中的 PHI

**這本 notebook 做什麼**：把一個小型語言模型（Gemma 4 E2B）訓練成「病歷守門人」。它讀一份中英夾雜的繁體中文病歷，輸出一份 JSON 清單，列出病歷中的個人可識別資訊（PHI，例如姓名、病歷號、電話）。接著我們再用 Python 依清單把病歷遮蔽成 `[NAME]`、`[DATE]` 這類標籤。

比喻：基礎模型像剛畢業、讀過很多書但沒看過這家醫院病歷格式的實習醫學生；微調（fine-tuning）像讓他在帶教老師指導下看幾百份標好答案的病歷，練習「這種寫法就是 PHI」。

> ## 警示：只能使用合成資料，不可上傳真實病歷
> Colab 是 Google 的雲端機器。你在這裡載入、上傳或貼上的任何資料，都會經過 Google 伺服器。**請勿把任何真實病歷、真實病人資料放進這本 notebook。** 本教學只使用「完全虛構的合成病歷」。真實病歷的去識別化需要院方核准與 IRB，詳見最後一節。

**預估時間**：待實測（安裝、載入模型、訓練、前後評估、匯出 GGUF 各階段的 T4 實際時間，跑過一次後再補）。

**使用方式**：選單 Runtime（執行階段）> Change runtime type（變更執行階段類型）> 選 T4 GPU，然後由上往下逐格執行。每個程式格前面都有一段說明。

本 notebook 以 Unsloth 官方 Gemma 4 E2B notebook 為底改寫（Unsloth 為 LGPL-3.0 授權）。

## 步驟 1：確認有拿到 GPU

下面這一行 `!nvidia-smi` 是在問機器「你有幾張顯示卡、記憶體多大、現在誰在用」。比喻：開刀前先確認手術室有排到，不要等到切皮才發現沒房間。

- 看到 `Tesla T4`、約 15 GB 記憶體 → 正常。
- 出現找不到指令或錯誤 → 回到選單把執行階段類型改成 T4 GPU。

In [ ]:
!nvidia-smi

## 步驟 2：安裝套件（沿用官方 cell，不要改）

這一格在安裝訓練需要的工具箱：Unsloth（讓訓練省記憶體、跑得快的工具）、TRL（提供 SFTTrainer 這個訓練器）、bitsandbytes（4-bit 壓縮）等。

逐行大意：
- `%%capture`：把安裝時一大串訊息收起來，畫面乾淨。
- `if "COLAB_" ...`：判斷是不是在 Colab。不是就單純 `pip install unsloth`；是的話就依 Colab 目前的 PyTorch 版本挑相容的 xformers 版本。
- `!pip install ...`：實際安裝。版本號是官方釘死的，因為套件之間版本不合就像藥物交互作用，可能出問題。
- 最後 `torch._dynamo.config.recompile_limit = 64`：放寬編譯快取上限，避免訓練中途被打斷。

安裝需要數分鐘。官方版本號會隨時間更新；若這格失敗，先到官方 notebook 複製最新的安裝 cell。

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.35")
    if str(torch.version.cuda).startswith("13") and xformers.endswith("0.0.35"): xformers = "https://download.pytorch.org/whl/cu130/xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl"
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

## 步驟 2b：補裝 timm（官方 cell）

Gemma 4 本身也能看圖、聽聲音，`timm` 是它視覺 / 音訊部分的零件。我們這次只訓練文字，但官方載入流程需要它在場，所以照裝。比喻：器械包裡有一把這次手術用不到、但器械清點規定要在的剪刀。

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

## 步驟 3：載入合成病歷資料集

這一格從本站作者的 GitHub 公開資料集 repo 下載三份 jsonl：

- `train`（訓練集）：給模型練習，像課本習題。
- `validation`（驗證集）：訓練途中拿來檢查有沒有學歪，像隨堂小考。
- `test`（測試集）：最後才打開，像國考。**訓練過程完全沒看過**，分數才有意義。

每筆資料欄位：`id`、`specialty`（科別）、`note_type`（病歷類型）、`style`（寫作風格）、`text`（病歷本文）、`phi`（答案清單，每項含 `type`、`text`、`start`、`end`）、`deid_text`（已遮蔽版本）。

下面 `DATA_BASE` 是資料集的網址（GitHub raw）。如果你的網路連不到 GitHub，可以從本站下載 zip、解壓後改用下一格的「上傳 jsonl」備援。

接著我們把每筆資料轉成 chat 格式 `{"messages":[system,user,assistant]}`。其中：
- `system` = 任務說明（逐字複製自專案的 `synth/build_sft.py`，訓練與評估必須用同一句，否則等於換了題目）。
- `user` = 病歷本文。
- `assistant` = 標準答案，一個 JSON 字串 `{"phi":[{"type":...,"text":...}]}`。

`target()`、`to_chat()` 也是逐字複製，不做任何改寫。

In [ ]:
import json
from datasets import load_dataset, DatasetDict

DATA_BASE = "https://raw.githubusercontent.com/fireman333/tw-synthetic-clinical-notes-phi/main/data"

# train / validation / test 三個 split，各是一個 jsonl 檔
raw = load_dataset("json", data_files={
    "train": f"{DATA_BASE}/train.jsonl",
    "validation": f"{DATA_BASE}/validation.jsonl",
    "test": f"{DATA_BASE}/test.jsonl",
})
print(raw)

# ---- 以下逐字複製自 synth/build_sft.py ----
SYSTEM = (
    "你是病歷去識別化助手。找出病歷中所有個人可識別資訊（PHI），"
    "類型限 NAME, DOCTOR, MRN, IDNO, DATE, PHONE, ADDRESS, HOSPITAL。"
    "只輸出 JSON：{\"phi\": [{\"type\": 類型, \"text\": 原文字串}]}，依出現順序，重複出現也要列出。"
)


def target(rec):
    return json.dumps({"phi": [{"type": p["type"], "text": p["text"]} for p in rec["phi"]]}, ensure_ascii=False)


def to_chat(rec):
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": rec["text"]},
        {"role": "assistant", "content": target(rec)},
    ]}
# ---- 複製結束 ----

## 步驟 3b（備援）：改用本機上傳的 jsonl

如果資料集還沒公開、或 HF 連不上，可以改用這格：自己從電腦上傳 `train.jsonl`、`validation.jsonl`、`test.jsonl`（需為含 `text`、`phi` 等欄位的「gold」格式，每行一筆 JSON，**不是**已經轉成 messages 的版本）。

預設 `USE_UPLOAD = False`，這格什麼都不做。要用時改成 `True`，執行後會跳出選檔視窗。再次提醒：只能上傳合成資料。

In [ ]:
USE_UPLOAD = False  # 要用本機檔案時改成 True

if USE_UPLOAD:
    from google.colab import files
    from datasets import Dataset
    uploaded = files.upload()  # 一次選 train.jsonl, validation.jsonl, test.jsonl
    splits = {}
    for fname in uploaded:
        name = fname.split(".")[0]  # train / validation / test
        splits[name] = Dataset.from_list([json.loads(l) for l in open(fname, encoding="utf-8")])
    raw = DatasetDict(splits)
    print(raw)

## 步驟 3c：把資料轉成 messages，並確認筆數

這一格做三件事：
1. 把 train / validation 轉成 chat 格式。`remove_columns` 會丟掉原本的欄位（包含病歷的 `text`），因為等一下訓練器會用「套完模板的文字」建立一個新的 `text` 欄位，不先清掉會混在一起。
2. 另外保留 test 的**原始版本** `gold_test`（含標準答案 `phi`），評估時要拿它對答案。比喻：考卷發給學生，答案卷鎖在老師抽屜。
3. 印出各集筆數與第一筆的標準答案，目視確認格式正確。

In [ ]:
train_msgs = raw["train"].map(to_chat, remove_columns=raw["train"].column_names)
val_msgs = raw["validation"].map(to_chat, remove_columns=raw["validation"].column_names)
gold_test = [dict(r) for r in raw["test"]]  # 評估用：保留原始欄位與標準答案

print({k: len(v) for k, v in raw.items()})
print(train_msgs[0]["messages"][2]["content"])  # 第一筆的標準答案

## 步驟 4：載入模型 Gemma 4 E2B

這一格把「還沒受過這項訓練的實習醫學生」請進來：`unsloth/gemma-4-E2B-it`（E2B 是約 2B 有效參數的小模型，`it` 表示已經學過聽指令）。

逐行大意：
- `from unsloth import FastModel`：Unsloth 的載入工具。
- `model_name`：要載入哪個模型。
- `dtype = None`：數字精度讓程式自動選（T4 不支援 bfloat16，會自動用 float16）。
- `max_seq_length = 2048`：一次最多讀 2048 個 token（約一份病歷加答案的長度上限）。比喻：病歷夾的厚度上限，太厚的會被截斷。
- `load_in_4bit = False`：照官方預設，用較高精度，品質較好但吃記憶體。**如果 T4（約 15 GB）記憶體不夠、出現 CUDA out of memory，就改成 `True`**，模型會被壓縮，像把 X 光片轉成較小的壓縮檔，省空間、略損細節。
- `full_finetuning = False`：不重練整個模型，只練後面加的小模組（LoRA，下一步說明）。

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, # None for auto detection
    max_seq_length = 2048, # 一份病歷 + 答案的長度上限
    load_in_4bit = False,  # T4 記憶體不夠（CUDA OOM）時改成 True
    full_finetuning = False,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

## 步驟 5：加上 LoRA 配接器（只訓練語言層）

LoRA 的概念：不改動整個模型（太貴、太慢），而是在旁邊貼上一小疊「便利貼」，只訓練便利貼。比喻：不重寫整本教科書，只在重點頁貼註解。

各參數的醫學比喻：

- `finetune_vision_layers = False`：不訓練「看圖」的部分。我們只處理文字，就像讀病歷不需要訓練放射科的讀片能力。
- `finetune_language_layers = True`：訓練「語言」的部分，這是這次任務的核心。要求指定為只訓練語言層。
- `finetune_attention_modules = True`：訓練注意力模組，負責「讀到這段時要回頭看哪裡」，像讀病歷時知道主訴要和現病史對照。
- `finetune_mlp_modules = True`：訓練 MLP 模組，負責「把看到的資訊消化、整理」，像查房後在腦中整合出 assessment。
- `r = 16`：便利貼的「厚度」（秩）。越大越能學複雜規則，但可能死背（過度擬合，overfitting）。像實習醫學生的筆記頁數：太少記不完，太多變成背題庫。
- `lora_alpha = 16`：便利貼的「音量旋鈕」，控制註解影響力的大小。慣例是設成與 `r` 相同。
- `lora_dropout = 0`：訓練時隨機遮住一部分便利貼來防止死背。這裡設 0 表示不遮（官方設定）。
- `bias = "none"`：不額外訓練偏移量，維持簡單。
- `random_state = 3407`：亂數種子，固定它就能讓別人重現同一個結果，像統計裡的 `set.seed`。

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 16,           # Larger = higher accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

## 步驟 6：套用 Gemma 4 的對話模板，並把資料變成「文字」

不同模型有各自的對話格式，像不同醫院的病歷表頭不一樣。`get_chat_template(tokenizer, chat_template = "gemma-4")` 讓 tokenizer（把文字切成 token 的工具）用 Gemma 4 的格式，大致長這樣：

```
<bos><|turn>user
Hello<turn|>
<|turn>model
Hey there!<turn|>
```

接著 `formatting_prompts_func` 把每筆 messages 轉成一段完整文字，存在 `text` 欄位。`.removeprefix('<bos>')` 是因為訓練器之後會自己加一個開頭符號 `<bos>`，這裡先去掉避免重複（官方作法）。

最後印出第一筆，目視檢查：應該看得到 system 說明、病歷、JSON 答案，而且只有一個 `<bos>`（或沒有）。

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

def formatting_prompts_func(examples):
   convos = examples["messages"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
   return { "text" : texts, }

train_ds = train_msgs.map(formatting_prompts_func, batched = True)
val_ds = val_msgs.map(formatting_prompts_func, batched = True)
print(train_ds[0]["text"])

## 步驟 7：評估用的函式（逐字複製自 `synth/evaluate.py`）

下面的 `parse` 和 `score` 是專案評估程式的**原封不動複製**，不 import 本機檔案（Colab 上沒有那些檔案）。這樣 Colab 上的分數和 Mac 上的分數用同一把尺。

- `parse(raw)`：把模型輸出的文字解析成 PHI 清單。模型若沒輸出合法 JSON，回傳 `None`，記為格式失敗。
- `score(golds, preds)`：計算指標。
  - **span_recall（主指標）**：標準答案中的每個 PHI 字串，只要出現在模型預測的集合裡就算抓到。因為遮蔽是「把預測字串的所有出現處都換掉」，漏抓一個就代表真的洩漏一個。比喻：像篩檢的敏感度（sensitivity），漏掉的就是 false negative，在去識別化是最不能接受的錯誤。
  - **95% CI**：用 bootstrap 重抽樣 2000 次估計的信賴區間，顯示「測試集只有這麼多份病歷，分數大概會晃多大」。
  - `precision_unique`：預測出的不重複字串中，有多少真的是 PHI（像陽性預測值）。
  - `clean_note_rate`：整份病歷的 PHI 全抓到的比例。
  - `valid_json_rate`：輸出是合法 JSON 的比例。
  - `recall_by_type`：各 PHI 類型的 recall。

這一格只定義函式，不會輸出東西。

In [ ]:
import json, random, re
from collections import defaultdict

# ---- 以下逐字複製自 synth/evaluate.py ----
def parse(raw):
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)
    raw = re.sub(r"<\|channel>.*?<channel\|>", "", raw, flags=re.S)
    m = re.search(r"\{.*\}", raw, flags=re.S)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return [p for p in obj.get("phi", []) if isinstance(p, dict) and isinstance(p.get("text"), str)]
    except (json.JSONDecodeError, AttributeError):
        return None


def score(golds, preds):
    per_note, by_type = [], defaultdict(lambda: [0, 0])
    tp_u = pred_u = 0
    for g, p in zip(golds, preds):
        ptexts = {x["text"] for x in (p or []) if x["text"].strip()}
        caught = [s["text"] in ptexts for s in g["phi"]]
        for s, c in zip(g["phi"], caught):
            by_type[s["type"]][0] += c
            by_type[s["type"]][1] += 1
        gtexts = {s["text"] for s in g["phi"]}
        tp_u += len(ptexts & gtexts)
        pred_u += len(ptexts)
        per_note.append((sum(caught), len(caught)))
    rec = sum(a for a, _ in per_note) / sum(b for _, b in per_note)
    rng = random.Random(0)
    boots = []
    for _ in range(2000):
        sample = [per_note[rng.randrange(len(per_note))] for _ in per_note]
        boots.append(sum(a for a, _ in sample) / max(1, sum(b for _, b in sample)))
    boots.sort()
    prec = tp_u / pred_u if pred_u else 0.0
    return {
        "n_notes": len(golds),
        "span_recall": round(rec, 4),
        "span_recall_95ci": [round(boots[49], 4), round(boots[1949], 4)],
        "precision_unique": round(prec, 4),
        "f1": round(2 * prec * rec / (prec + rec), 4) if prec + rec else 0.0,
        "clean_note_rate": round(sum(a == b for a, b in per_note) / len(per_note), 4),
        "valid_json_rate": round(sum(p is not None for p in preds) / len(preds), 4),
        "recall_by_type": {t: round(a / b, 4) for t, (a, b) in sorted(by_type.items())},
    }
# ---- 複製結束 ----

## 步驟 8：產生預測的輔助函式，並先量一次「微調前」的分數

要比較前後，就要先量一次基準線（baseline）。這一格定義 `predict_notes()`：對測試集每份病歷產生預測。

逐行大意：
- 把 `[system, user]` 兩段訊息套上模板，`add_generation_prompt = True` 表示「輪到模型說話了」。
- `max_new_tokens = 500`：最多生成 500 個 token。
- `do_sample = False`：greedy（貪婪）解碼，每一步都選機率最高的字，不抽籤。好處是同樣輸入每次輸出相同，評估才可重現；不使用官方聊天示範那組 `temperature / top_p / top_k` 隨機取樣。
- 只解碼「新生成的部分」（`outputs[0][input_len:]`），不把題目一起讀回來。
- `parse()` 把輸出轉成 PHI 清單。

`EVAL_LIMIT` 可以先設小一點（例如 20）試跑，確認流程通了再設成 `None` 跑完整測試集。逐筆生成在 T4 上可能較慢，實際時間待實測。

**關於「微調前」**：此刻 LoRA 便利貼剛貼上去、尚未訓練（其初始值不改變模型輸出），所以現在量到的就是基礎模型的分數。請務必在訓練之前執行這一格，並把結果存在 `base_result`。

`enable_thinking = False` 是額外的 template 參數，沿用專案 Mac 端 evaluate.py 的設定，要關掉思考模式；此參數在 Colab 版 Gemma 4 template 是否生效待實測。

In [ ]:
import time

# Gemma 4 的 processor 要求 content 是「片段清單」格式：[{"type": "text", "text": ...}]
# （它同時支援圖片、音訊）。直接傳字串會出現 TypeError: string indices must be integers。
def as_parts(msgs):
    return [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in msgs]

EVAL_LIMIT = None  # 先試跑可設 20；None = 跑完整 test set

def predict_notes(golds, max_new_tokens = 500):
    preds, raws = [], []
    t0 = time.time()
    for i, g in enumerate(golds):
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": g["text"]}]
        inputs = tokenizer.apply_chat_template(
            as_parts(msgs),
            add_generation_prompt = True, # Must add for generation
            return_tensors = "pt",
            tokenize = True,
            return_dict = True,
            enable_thinking = False,      # 待實測：此參數在此 template 是否生效
        ).to("cuda")
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample = False,            # greedy：結果可重現
        )
        text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)
        raws.append(text)
        preds.append(parse(text))
        if (i + 1) % 10 == 0:
            print(f"{i + 1}/{len(golds)}  {time.time() - t0:.0f}s", flush = True)
    return preds, raws

golds = gold_test[:EVAL_LIMIT]
base_preds, base_raws = predict_notes(golds)
base_result = score(golds, base_preds)
print(json.dumps(base_result, ensure_ascii = False, indent = 2))

## 步驟 9：設定訓練器（SFTTrainer）

SFTTrainer 是「帶教老師」，負責監督式微調（SFT：給題目＋標準答案，讓模型模仿）。

逐項說明（每項都有醫學比喻）：
- `per_device_train_batch_size = 2`：一次看 2 份病歷。像一次查 2 床。
- `gradient_accumulation_steps = 4`：看完 4 小批再一起更新一次，等效一次看 2 x 4 = 8 份。像累積 8 床的查房心得再統一修正筆記，省記憶體又保有大批次的穩定。
- `num_train_epochs = 2`：整套訓練資料看 2 輪。像同一本題庫刷兩遍。
- `learning_rate = 1e-4`：每次修正的幅度。太大會學歪，太小學不動。官方示範是 2e-4，這裡依需求用 1e-4，較保守。
- `warmup_steps = 5`：前 5 步先小幅度起步，像手術前先暖身。
- `logging_steps = 10`：每 10 步印一次 loss（錯誤程度，越小越好）。
- `eval_strategy = "epoch"` 與 `per_device_eval_batch_size = 2`：每輪結束用 validation 小考一次，印出 eval loss。這是 Hugging Face transformers 的標準訓練參數（非 Unsloth 專屬）；其在 T4 上的記憶體影響待實測。
- `eval_dataset = val_ds`：隨堂小考用的資料。
- `optim = "adamw_8bit"`、`weight_decay = 0.001`、`lr_scheduler_type = "linear"`、`seed = 3407`：優化器與排程，沿用官方。
- `report_to = "none"`：不把紀錄送到外部追蹤服務。

注意：官方示範用 `max_steps = 60` 求快；這裡改用 `num_train_epochs = 2` 跑完整訓練。

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds, # 隨堂小考
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 1e-4,
        logging_steps = 10,
        eval_strategy = "epoch",
        per_device_eval_batch_size = 2,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

## 步驟 10：只對「答案」計算學習成績（train_on_responses_only）

預設情況下，模型會連「題目（病歷本文）」也一起學著背。但我們只希望它學會「看到病歷後該輸出什麼」。`train_on_responses_only` 把題目部分遮起來、不計入損失（loss），只有 assistant 的 JSON 答案才算分。比喻：批改作業只看答案欄，不批改題目抄寫。

Unsloth 會自動從模板偵測哪段是題目、哪段是答案，所以不需要手動指定（官方作法）。

套用完會再用下一格驗證遮罩是否正確。

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)

## 步驟 10b：檢查遮罩有沒有生效

這一格先印出完整輸入（題目＋答案），再印出「被計分」的部分：被遮掉的位置（標籤為 -100）換成空白，所以只應剩 JSON 答案。若還看到病歷本文，代表遮罩沒生效。

In [ ]:
print(tokenizer.decode(trainer.train_dataset[0]["input_ids"]))
print("=" * 40)
# 被遮掉的位置（-100）換成空白，只剩計分的答案部分
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " "))

## 步驟 11：訓練前的記憶體基準（官方 cell）

記下訓練開始前 GPU 用了多少記憶體，訓練完才能算出「訓練本身多吃了多少」。像開刀前先量基礎生命徵象，術後才有得比較。

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

## 步驟 12：開始訓練

`trainer.train()` 就是正式開始練習。畫面會每 10 步印一次 training loss，每輪結束印一次 validation loss。

怎麼看：
- training loss 大致往下 → 正常學習中。
- validation loss 若先降後升，可能開始死背（過度擬合）。
- 若中斷想接續，改成 `trainer.train(resume_from_checkpoint = True)`。

預估時間：待實測（取決於資料筆數與 T4 速度）。Colab 閒置太久會斷線，請保持分頁開著。

In [ ]:
trainer_stats = trainer.train()

## 步驟 13：訓練後的記憶體與時間統計（官方 cell）

印出訓練花了多久、尖峰記憶體多少，以及訓練本身多吃的記憶體。把這些數字填回開頭的「預估時間：待實測」，就是這本 notebook 的實測紀錄。

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## 步驟 14：微調後評估，並與「微調前」比較

這一格用**完全相同的測試集、相同的解碼設定（greedy、500 tokens）、相同的 `parse` / `score`**，量微調後的分數，再和步驟 8 的 `base_result` 並排印出：

- `span_recall` 與 95% CI（主指標）
- `valid_json_rate`（輸出是否為合法 JSON）
- 各類型 recall（NAME、DOCTOR、MRN……哪一類容易漏）

**怎麼解讀 CI（請照這個措辭，不要寫過頭）：**
- 兩者 95% CI **不重疊**：可以說「在這份測試集上，微調後的 span_recall 高於微調前，兩者信賴區間不重疊」。
- 兩者 95% CI **有重疊**：只能說「在這份測試集上未顯示明確差異」。**不可寫成「兩者一樣」或「沒有差別」**：信賴區間重疊代表樣本不足以區分，不代表真的相同。
- 兩條 CI 各自獨立估計，「不重疊」是保守的判準；「有重疊」並不等於「沒差異」，要證明相同需要另外設計（例如預設 margin 的非劣性檢定）。
- 這些分數只代表在**合成資料**上的表現。

In [ ]:
ft_preds, ft_raws = predict_notes(golds)
ft_result = score(golds, ft_preds)

def show(name, r):
    lo, hi = r["span_recall_95ci"]
    print(f"[{name}] n_notes={r['n_notes']}  span_recall={r['span_recall']:.3f} (95% CI {lo:.3f}-{hi:.3f})  "
          f"valid_json_rate={r['valid_json_rate']:.3f}  precision_unique={r['precision_unique']:.3f}  "
          f"clean_note_rate={r['clean_note_rate']:.3f}")

show("fine-tune 前", base_result)
show("fine-tune 後", ft_result)

print("\n各類型 recall：")
types = sorted(set(base_result["recall_by_type"]) | set(ft_result["recall_by_type"]))
print(f"{'type':10s}{'前':>8s}{'後':>8s}")
for t in types:
    print(f"{t:10s}{base_result['recall_by_type'].get(t, float('nan')):8.3f}{ft_result['recall_by_type'].get(t, float('nan')):8.3f}")

b_lo, b_hi = base_result["span_recall_95ci"]
f_lo, f_hi = ft_result["span_recall_95ci"]
print()
if f_lo > b_hi:
    print("在這份合成測試集上，微調後 span_recall 高於微調前，兩者 95% CI 不重疊。")
elif b_lo > f_hi:
    print("在這份合成測試集上，微調後 span_recall 低於微調前，兩者 95% CI 不重疊。")
else:
    print("兩者 95% CI 有重疊：在這份合成測試集上未顯示明確差異（不代表兩者相同，樣本可能不足以區分）。")

## 步驟 15：實際試用，並用輸出的清單遮蔽病歷

最後做一次完整流程：貼一份**全新的合成病歷**（下面這份是虛構的，姓名、號碼皆非真人），讓模型找出 PHI，再用 Python 把每個 PHI 字串換成 `[TYPE]` 標籤。

遮蔽邏輯：
1. `parse()` 取得 PHI 清單。
2. 依字串長度由長到短排序再取代，避免短字串先被換掉、破壞長字串（像先處理整串地址，再處理其中的門牌號碼）。
3. 把每個出現處換成 `[類型]`。

重要：遮蔽只會去掉模型「抓到」的字串，**漏抓的 PHI 會原樣留在輸出裡**。這就是為什麼我們用 span_recall 當主指標。你可以把 `new_note` 換成自己編的合成病歷再試。**請勿貼真實病歷。**

In [ ]:
new_note = (
    "【急診病歷】王小明 32y/o male，MRN 4821937，身分證 K187654320。"
    "2024/03/15 因 RLQ pain 18 hrs 到青嵐綜合醫院急診，BT 38.1°C, HR 98, WBC 14,200。"
    "CT: dilated appendix with periappendiceal fat stranding，impression: acute appendicitis，已會診一般外科 吳志遠 醫師。"
    "家屬王美玲 (姊姊) 電話 0912-345-678，住 竹田縣星湖鄉月橋路 88 號。"
)

msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": new_note}]
inputs = tokenizer.apply_chat_template(
    as_parts(msgs), add_generation_prompt = True, return_tensors = "pt", tokenize = True, return_dict = True,
    enable_thinking = False,
).to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 500, do_sample = False)
raw_out = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)
print("模型輸出：", raw_out)

phi = parse(raw_out) or []

def mask_note(text, phi_list):
    # 由長到短取代；同一字串出現多次只取第一個類型
    type_of = {}
    for p in phi_list:
        type_of.setdefault(p["text"], p.get("type", "PHI"))
    for s in sorted(type_of, key = len, reverse = True):
        if s.strip():
            text = text.replace(s, f"[{type_of[s]}]")
    return text

print("\n遮蔽後：")
print(mask_note(new_note, phi))

## 步驟 16：匯出成 GGUF，帶回 Mac 用 Ollama 跑

GGUF 是 llama.cpp / Ollama 使用的單一檔案模型格式。`save_pretrained_gguf` 會把「基礎模型 + 我們訓練的 LoRA」合併後轉成 GGUF，量化方式 `Q8_0`（8-bit，檔案較大但品質損失小）。官方說明目前只支援 `Q8_0`、`BF16`、`F16`。

這一格預設 `if False` 不執行（匯出需要時間與磁碟空間）。確定要匯出時改成 `True`。

匯出後，用下一格下載 `.gguf` 檔到你的電腦。檔案可能數 GB，下載較久；也可以改存到 Google Drive 再下載。

**標註：Mac 上本專案實測：mlx 路線需先 llama.cpp 轉 GGUF；Unsloth 產的 GGUF 能否被 Ollama 0.35 載入待實測（E5）。**

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "chartkeeper_gemma4_e2b",
        tokenizer,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

## 步驟 16b：下載 .gguf 並產生 Modelfile

這一格找出剛匯出的 `.gguf`，用 Colab 的 `files.download` 傳到你的電腦（瀏覽器會開始下載），同時產生一份 `Modelfile` 一起下載。Modelfile 像是「這個模型的處方箋」：告訴 Ollama 要載入哪個檔、系統提示是什麼、取樣要不要隨機。

其中 `SYSTEM` 沿用與訓練相同的任務說明。`TEMPLATE`（Gemma 4 對話模板）這裡**不寫**：Ollama 能否自動套用正確模板待實測，若輸出怪異，需要手動補 TEMPLATE。

**把檔案放到 Mac 後的指令**（不在 Colab 裡跑，在 Mac 終端機）：

```bash
# 假設 .gguf 與 Modelfile 都在 ~/Downloads/chartkeeper/
cd ~/Downloads/chartkeeper
# Modelfile 第一行應為：FROM ./<你的檔名>.gguf
ollama create chartkeeper -f Modelfile
ollama run chartkeeper
```

再提醒一次：Unsloth 產的 GGUF 能否被 Ollama 0.35 載入待實測（E5）。載入失敗時，請以 llama.cpp 直接跑，並回報錯誤訊息。

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob("chartkeeper_gemma4_e2b*/*.gguf") + glob.glob("chartkeeper_gemma4_e2b*.gguf")
print("找到的 GGUF：", gguf_files)

if gguf_files:
    gguf_path = gguf_files[0]
    modelfile = f'''FROM ./{os.path.basename(gguf_path)}
PARAMETER temperature 0
SYSTEM """{SYSTEM}"""
'''
    open("Modelfile", "w", encoding="utf-8").write(modelfile)
    print(modelfile)
    files.download("Modelfile")
    files.download(gguf_path)  # 檔案可能數 GB，下載需要時間
else:
    print("尚未匯出 GGUF：請先把上一格的 if False 改成 if True 並執行。")

## 這本 notebook 的限制（請務必讀完）

1. **分數只代表在合成資料上的表現。** 測試集是用程式從假資料填出來的，格式與寫法比真實病歷單純；在這裡得到的 span_recall，不能推論到真實病歷。
2. **不代表可用於真實病歷。** 本 notebook 產出的模型不是經過驗證的臨床去識別化工具，不可用來處理、分享或發表真實病人資料。
3. **真實病歷需要院方核准與 IRB。** 若要用真實病歷做研究，必須取得院方同意與倫理審查（IRB）核准，並在院內合規的環境中處理，**不可上傳到 Colab 等外部雲端**。
4. 測試集筆數有限，95% CI 可能很寬；CI 重疊時不能下「兩者相同」的結論。
5. 預估時間、T4 記憶體是否足夠、`enable_thinking` 是否生效、GGUF 能否被 Ollama 0.35 載入（E5），皆標為待實測。
6. 作者為醫學系畢業的教學作者，本教材為學習用途，不構成醫療或法律建議。